<a href="https://colab.research.google.com/github/RishithaErva/Night2Day-Segmentation/blob/main/INLP_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers scikit-learn pandas torch tqdm --quiet

In [2]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# Change this to wherever you put your folder in Drive
DATA_DIR = "/content/drive/MyDrive/INLP_datasets"

# Check all files are visible
files_needed = [
    "QAEvasion.csv",
    "raw_train_biden.csv", "raw_val_biden.csv", "raw_test_biden.csv",
    "raw_train_trump.csv", "raw_val_trump.csv", "raw_test_trump.csv",
    "raw_train_bernie.csv","raw_val_bernie.csv","raw_test_bernie.csv",
]

print("Checking files:")
for f in files_needed:
    path = os.path.join(DATA_DIR, f)
    status = "✓ found" if os.path.exists(path) else "✗ MISSING"
    print(f"  {status}  {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking files:
  ✓ found  QAEvasion.csv
  ✓ found  raw_train_biden.csv
  ✓ found  raw_val_biden.csv
  ✓ found  raw_test_biden.csv
  ✓ found  raw_train_trump.csv
  ✓ found  raw_val_trump.csv
  ✓ found  raw_test_trump.csv
  ✓ found  raw_train_bernie.csv
  ✓ found  raw_val_bernie.csv
  ✓ found  raw_test_bernie.csv


In [9]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 3 — Imports + Config                                  │
# └─────────────────────────────────────────────────────────────┘

import os, random, warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertModel, DistilBertTokenizer,
                          get_linear_schedule_with_warmup)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix)
warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

# ── Hyperparameters ────────────────────────────────────────────
BERT_MODEL      = "distilbert-base-uncased"  # lighter + faster than BERT
MAX_LEN         = 256    # Q+A pairs (evasion task)
STANCE_MAX_LEN  = 128    # tweets are short
BATCH_SIZE      = 16     # lower to 8 if CUDA out of memory
EPOCHS          = 5
LR              = 2e-5
DROPOUT         = 0.3
WARMUP_RATIO    = 0.1
BEST_MODEL_PATH = "best_model.pt"

# ── Labels ─────────────────────────────────────────────────────
EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}

STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

# ── Dataset file paths ─────────────────────────────────────────
EVASION_CSV       = f"{DATA_DIR}/QAEvasion.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",
                     f"{DATA_DIR}/raw_train_trump.csv",
                     f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",
                     f"{DATA_DIR}/raw_val_trump.csv",
                     f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv",
                     f"{DATA_DIR}/raw_test_trump.csv",
                     f"{DATA_DIR}/raw_test_bernie.csv"]

print("\n✓ Cell 3 complete — config loaded")

Device : cuda
GPU    : Tesla T4

✓ Cell 3 complete — config loaded


In [10]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 4 — Data Loading                                      │
# └─────────────────────────────────────────────────────────────┘

def map_evasion_label(raw):
    """
    The QAEvasion dataset has 9 fine-grained labels.
    We map them to 3 coarse labels as per the project proposal:
      1.x  → Non-Evasive       (clear reply)
      2.3  → Partially Evasive (ambivalent reply)
      2.x  → Evasive           (non-reply / dodging)
    """
    raw = str(raw).strip()
    if raw.startswith("1."):
        return "Non-Evasive"
    elif raw == "2.3 Partial/half-answer":
        return "Partially Evasive"
    else:
        return "Evasive"

def load_evasion(filepath):
    print(f"\n{'─'*55}")
    print(f"EVASION  Loading: {filepath}")
    df = pd.read_csv(filepath)
    print(f"  Raw rows: {len(df)}")

    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)

    # Build input: [Q] question [A] gpt summary of answer
    df["text"] = (
        "[Q] " + df["question"].fillna("").str.strip() +
        " [A] " + df["gpt3.5_summary"].fillna("").str.strip()
    )
    df = df.dropna(subset=["text","label_id"]).reset_index(drop=True)

    print("  Label distribution:")
    for lbl in EVASION_LABELS:
        n   = (df["coarse_label"] == lbl).sum()
        bar = "█" * (n // 50)
        print(f"    {lbl:22s}: {n:4d}  {bar}")

    # Stratified 70 / 15 / 15 split
    X, y = df["text"].tolist(), df["label_id"].tolist()
    Xtr,Xtmp,ytr,ytmp = train_test_split(X,y, test_size=0.30, random_state=SEED, stratify=y)
    Xva,Xte,yva,yte   = train_test_split(Xtmp,ytmp, test_size=0.50, random_state=SEED, stratify=ytmp)
    print(f"  Train:{len(Xtr)}  Val:{len(Xva)}  Test:{len(Xte)}")
    return (Xtr,ytr),(Xva,yva),(Xte,yte)

def load_stance_split(filepaths, split_name):
    print(f"\n{'─'*55}")
    print(f"STANCE [{split_name}]  Loading {len(filepaths)} files:")
    dfs = []
    for fp in filepaths:
        if os.path.exists(fp):
            df = pd.read_csv(fp)
            dfs.append(df)
            print(f"  ✓ {fp}: {len(df)} rows")
        else:
            print(f"  ✗ {fp}: NOT FOUND — skipping")

    if not dfs:
        raise FileNotFoundError(f"No stance files found for split='{split_name}'")

    df = pd.concat(dfs, ignore_index=True)
    df = df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"] = df["Stance"].map(STANCE_LABEL2ID).astype(int)

    # Build input: [TARGET] politician [TWEET] tweet text
    df["text"] = (
        "[TARGET] " + df["Target"].fillna("").str.strip() +
        " [TWEET] " + df["Tweet"].fillna("").str.strip()
    )
    df = df.dropna(subset=["text","label_id"]).reset_index(drop=True)

    print(f"  Combined: {len(df)} rows")
    print("  Label distribution:")
    for lbl in STANCE_LABELS:
        n   = (df["Stance"] == lbl).sum()
        bar = "█" * (n // 100)
        print(f"    {lbl:10s}: {n:5d}  {bar}")

    return df["text"].tolist(), df["label_id"].tolist()

# ── Load all data ───────────────────────────────────────────────
(Xetr,yetr),(Xeva,yeva),(Xete,yete) = load_evasion(EVASION_CSV)
Xstr,ystr = load_stance_split(STANCE_TRAIN_CSVS, "train")
Xsva,ysva = load_stance_split(STANCE_VAL_CSVS,   "val")
Xste,yste = load_stance_split(STANCE_TEST_CSVS,  "test")

print("\n✓ Cell 4 complete — all datasets loaded")



───────────────────────────────────────────────────────
EVASION  Loading: /content/drive/MyDrive/INLP_datasets/QAEvasion.csv
  Raw rows: 3448
  Label distribution:
    Non-Evasive           : 1540  ██████████████████████████████
    Partially Evasive     :   79  █
    Evasive               : 1829  ████████████████████████████████████
  Train:2413  Val:517  Test:518

───────────────────────────────────────────────────────
STANCE [train]  Loading 3 files:
  ✓ /content/drive/MyDrive/INLP_datasets/raw_train_biden.csv: 5806 rows
  ✓ /content/drive/MyDrive/INLP_datasets/raw_train_trump.csv: 6362 rows
  ✓ /content/drive/MyDrive/INLP_datasets/raw_train_bernie.csv: 5056 rows
  Combined: 17224 rows
  Label distribution:
    FAVOR     :  8347  ███████████████████████████████████████████████████████████████████████████████████
    AGAINST   :  8877  ████████████████████████████████████████████████████████████████████████████████████████

───────────────────────────────────────────────────────
STA

In [11]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 5 — Tokenizer + DataLoaders                           │
# └─────────────────────────────────────────────────────────────┘

print(f"Loading tokenizer: {BERT_MODEL} ...")
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

class TaskDataset(Dataset):
    def __init__(self, texts, labels, max_len, task_id):
        self.texts=texts; self.labels=labels
        self.max_len=max_len; self.task_id=task_id
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True,
                        padding="max_length", max_length=self.max_len,
                        return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":  torch.tensor(self.labels[idx], dtype=torch.long),
            "task_id": torch.tensor(self.task_id,     dtype=torch.long),
        }

def make_loader(X, y, task_id, max_len, shuffle):
    return DataLoader(TaskDataset(X,y,max_len,task_id),
                      batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2)

loaders = {
    "evasion_train": make_loader(Xetr,yetr, 0, MAX_LEN,         shuffle=True),
    "evasion_val":   make_loader(Xeva,yeva, 0, MAX_LEN,         shuffle=False),
    "evasion_test":  make_loader(Xete,yete, 0, MAX_LEN,         shuffle=False),
    "stance_train":  make_loader(Xstr,ystr, 1, STANCE_MAX_LEN,  shuffle=True),
    "stance_val":    make_loader(Xsva,ysva, 1, STANCE_MAX_LEN,  shuffle=False),
    "stance_test":   make_loader(Xste,yste, 1, STANCE_MAX_LEN,  shuffle=False),
}

print("\nDataLoader sizes:")
for k,v in loaders.items():
    print(f"  {k:20s}: {len(v.dataset):5d} samples  |  {len(v):4d} batches")

print("\n✓ Cell 5 complete — DataLoaders ready")

Loading tokenizer: distilbert-base-uncased ...

DataLoader sizes:
  evasion_train       :  2413 samples  |   151 batches
  evasion_val         :   517 samples  |    33 batches
  evasion_test        :   518 samples  |    33 batches
  stance_train        : 17224 samples  |  1077 batches
  stance_val          :  2193 samples  |   138 batches
  stance_test         :  2157 samples  |   135 batches

✓ Cell 5 complete — DataLoaders ready


In [12]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 6 — Model                                             │
# └─────────────────────────────────────────────────────────────┘

class GatingNetwork(nn.Module):
    """
    Target-aware gating (Song et al. 2019 — cited in your proposal).
    Learns which features in the DistilBERT vector matter for each task.
    """
    def __init__(self, H, dropout):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(H, H), nn.Tanh(),
            nn.Linear(H, H), nn.Sigmoid()
        )
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.gate(x) * x)

class ClassHead(nn.Module):
    def __init__(self, H, num_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.net(x)

class MultiTaskDistilBERT(nn.Module):
    """
    Architecture (your friend's design):

      DistilBERT Encoder   ← shared, always trains
            │
      Target-Aware Gating  ← one per task
            │
      ┌─────┴──────┐
      │            │
    Evasion      Stance
    Head         Head
    (3 cls)      (3 cls)

    During training: the head NOT being used is frozen.
    This prevents gradients from the wrong task corrupting
    a head's weights.
    """
    def __init__(self):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size   # 768

        self.evasion_gate = GatingNetwork(H, DROPOUT)
        self.stance_gate  = GatingNetwork(H, DROPOUT)
        self.evasion_head = ClassHead(H, len(EVASION_LABELS), DROPOUT)
        self.stance_head  = ClassHead(H, len(STANCE_LABELS),  DROPOUT)
        self.loss_fn      = nn.CrossEntropyLoss()

    def freeze_inactive_head(self, active_task):
        """Freeze the head that is NOT active right now."""
        if active_task == 0:   # training evasion → freeze stance
            for p in self.stance_head.parameters():  p.requires_grad = False
            for p in self.stance_gate.parameters():  p.requires_grad = False
            for p in self.evasion_head.parameters(): p.requires_grad = True
            for p in self.evasion_gate.parameters(): p.requires_grad = True
        else:                  # training stance → freeze evasion
            for p in self.evasion_head.parameters(): p.requires_grad = False
            for p in self.evasion_gate.parameters(): p.requires_grad = False
            for p in self.stance_head.parameters():  p.requires_grad = True
            for p in self.stance_gate.parameters():  p.requires_grad = True
        for p in self.encoder.parameters(): p.requires_grad = True  # always on

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True

    def forward(self, input_ids, attention_mask, task_id, labels=None):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]   # CLS token [batch, 768]
        task = int(task_id[0].item())

        if task == 0:
            logits = self.evasion_head(self.evasion_gate(cls))
        else:
            logits = self.stance_head(self.stance_gate(cls))

        loss = self.loss_fn(logits, labels) if labels is not None else None
        return loss, logits

model = MultiTaskDistilBERT().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Model: MultiTaskDistilBERT")
print(f"  Total parameters: {total:,}")

print("\n✓ Cell 6 complete — model built")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: MultiTaskDistilBERT
  Total parameters: 69,120,261

✓ Cell 6 complete — model built


In [13]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 7 — Evaluation helpers                                │
# └─────────────────────────────────────────────────────────────┘

@torch.no_grad()
def evaluate(loader, label_names):
    model.eval()
    model.unfreeze_all()
    all_preds, all_labels, total_loss = [], [], 0.0
    for batch in loader:
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device),
        )
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    n = len(loader)
    return {
        "loss"  : total_loss / n,
        "acc"   : accuracy_score(all_labels, all_preds),
        "f1"    : f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "prec"  : precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "rec"   : recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "report": classification_report(all_labels, all_preds,
                                        target_names=label_names, zero_division=0),
        "cm"    : pd.DataFrame(
            confusion_matrix(all_labels, all_preds,
                             labels=list(range(len(label_names)))),
            index=[f"True:{l}" for l in label_names],
            columns=[f"Pred:{l}" for l in label_names]
        ),
        "preds" : all_preds,
        "labels": all_labels,
    }

def print_eval(m, title):
    print(f"\n{'═'*60}")
    print(f"  {title}")
    print(f"{'═'*60}")
    print(f"  Loss             : {m['loss']:.4f}")
    print(f"  Accuracy         : {m['acc']:.4f}  ({m['acc']*100:.1f}%)")
    print(f"  Macro F1         : {m['f1']:.4f}")
    print(f"  Macro Precision  : {m['prec']:.4f}")
    print(f"  Macro Recall     : {m['rec']:.4f}")
    print(f"\n  Per-class breakdown:\n{m['report']}")
    print(f"  Confusion Matrix:\n{m['cm']}\n")

print("✓ Cell 7 complete — evaluation helpers ready")

✓ Cell 7 complete — evaluation helpers ready


In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 8 — TRAINING  (watch this run step by step)          │
# └─────────────────────────────────────────────────────────────┘

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
n_steps   = (len(loaders["evasion_train"]) + len(loaders["stance_train"])) * EPOCHS
n_warmup  = int(WARMUP_RATIO * n_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, n_warmup, n_steps)

best_avg_f1 = 0.0
history     = []

print("╔══════════════════════════════════════════════════════════════╗")
print("║          MULTI-TASK DISTILBERT TRAINING STARTED             ║")
print(f"║  Epochs:{EPOCHS}  Batch:{BATCH_SIZE}  LR:{LR}  Total steps:{n_steps}  ║")
print("╚══════════════════════════════════════════════════════════════╝\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_done       = 0

    # Build interleaved + shuffled batch list
    e_list = list(loaders["evasion_train"])
    s_list = list(loaders["stance_train"])
    combined = []
    for e, s in zip(e_list, s_list):
        combined.append(e); combined.append(s)
    short_len = min(len(e_list), len(s_list))
    longer = e_list if len(e_list) > len(s_list) else s_list
    for b in longer[short_len:]:
        combined.append(b)
    random.shuffle(combined)

    total = len(combined)
    print(f"\n{'━'*65}")
    print(f"  EPOCH {epoch} / {EPOCHS}     Total steps this epoch: {total}")
    print(f"{'━'*65}")

    for step, batch in enumerate(combined, 1):
        task      = int(batch["task_id"][0].item())
        task_name = "Evasion" if task == 0 else "Stance "

        # ── Freeze inactive head (your friend's key idea) ─────
        model.freeze_inactive_head(task)

        optimizer.zero_grad()
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        n_done       += 1
        avg_loss      = running_loss / n_done
        batch_acc     = (torch.argmax(logits,1) == batch["labels"].to(device)).float().mean().item()

        # ── Progress bar (updates every step) ─────────────────
        pct     = step / total
        filled  = int(40 * pct)
        bar     = "█" * filled + "░" * (40 - filled)
        print(f"\r  [{bar}] {step:4d}/{total}"
              f"  {task_name}"
              f"  Loss:{loss.item():.4f}"
              f"  AvgLoss:{avg_loss:.4f}"
              f"  BatchAcc:{batch_acc:.3f}",
              end="", flush=True)

        # ── Full line print every 50 steps ─────────────────────
        if step % 50 == 0:
            print(f"\n  [Step {step:4d}/{total}]"
                  f"  Task:{task_name}"
                  f"  Loss:{loss.item():.4f}"
                  f"  AvgLoss:{avg_loss:.4f}"
                  f"  BatchAcc:{batch_acc:.3f}")

    # ── End of epoch ───────────────────────────────────────────
    print(f"\n\n  Epoch {epoch} done. Avg train loss: {running_loss/n_done:.4f}")
    print("  Validating...")

    e_val = evaluate(loaders["evasion_val"], EVASION_LABELS)
    s_val = evaluate(loaders["stance_val"],  STANCE_LABELS)
    avg_f1 = (e_val["f1"] + s_val["f1"]) / 2
    is_best = avg_f1 > best_avg_f1

    print(f"\n  {'─'*60}")
    print(f"  EPOCH {epoch} VALIDATION SUMMARY")
    print(f"  {'─'*60}")
    print(f"  Evasion  │ Loss:{e_val['loss']:.4f} │ Acc:{e_val['acc']:.4f} │ Macro-F1:{e_val['f1']:.4f}")
    print(f"  Stance   │ Loss:{s_val['loss']:.4f} │ Acc:{s_val['acc']:.4f} │ Macro-F1:{s_val['f1']:.4f}")
    print(f"  {'─'*60}")
    print(f"  Avg Macro-F1 both tasks: {avg_f1:.4f}  {'  ← NEW BEST ✓' if is_best else ''}")
    print(f"  {'─'*60}")

    print(f"\n  Evasion per-class accuracy:")
    for i, lbl in enumerate(EVASION_LABELS):
        corr = sum(p==l==i for p,l in zip(e_val["preds"],e_val["labels"]))
        tot  = sum(l==i    for l   in e_val["labels"])
        pct  = corr/tot if tot else 0
        bar  = "█" * int(pct * 20) + "░" * (20 - int(pct * 20))
        print(f"    {lbl:22s}: {corr:4d}/{tot:4d}  [{bar}]  {pct:.1%}")

    print(f"\n  Stance per-class accuracy:")
    for i, lbl in enumerate(STANCE_LABELS):
        corr = sum(p==l==i for p,l in zip(s_val["preds"],s_val["labels"]))
        tot  = sum(l==i    for l   in s_val["labels"])
        pct  = corr/tot if tot else 0
        bar  = "█" * int(pct * 20) + "░" * (20 - int(pct * 20))
        print(f"    {lbl:10s}: {corr:4d}/{tot:4d}  [{bar}]  {pct:.1%}")

    history.append({
        "epoch":epoch, "train_loss":running_loss/n_done,
        "evasion_f1":e_val["f1"], "evasion_acc":e_val["acc"],
        "stance_f1":s_val["f1"],  "stance_acc":s_val["acc"],
        "avg_f1":avg_f1
    })

    if is_best:
        best_avg_f1 = avg_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"\n  ✓ BEST MODEL SAVED → {BEST_MODEL_PATH}  (F1={best_avg_f1:.4f})")

print(f"\n\n{'╔'+'═'*62+'╗'}")
print(f"║  TRAINING COMPLETE   Best avg F1: {best_avg_f1:.4f}                 ║")
print(f"{'╚'+'═'*62+'╝'}")

print("\nFull training history:")
print(pd.DataFrame(history).to_string(index=False))


╔══════════════════════════════════════════════════════════════╗
║          MULTI-TASK DISTILBERT TRAINING STARTED             ║
║  Epochs:5  Batch:16  LR:2e-05  Total steps:6140  ║
╚══════════════════════════════════════════════════════════════╝


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EPOCH 1 / 5     Total steps this epoch: 1228
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   50/1228  Evasion  Loss:1.1368  AvgLoss:0.7406  BatchAcc:0.062
  [Step   50/1228]  Task:Evasion  Loss:1.1368  AvgLoss:0.7406  BatchAcc:0.062
  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  100/1228  Stance   Loss:0.6976  AvgLoss:0.7393  BatchAcc:0.438
  [Step  100/1228]  Task:Stance   Loss:0.6976  AvgLoss:0.7393  BatchAcc:0.438
  [████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  150/1228  Stance   Loss:0.6993  AvgLoss:0.7522  BatchAcc:0.375
  [Step  150/1228]  Task:Stance   Loss:0.6993  AvgLoss:0.7522  BatchAcc:0.375
  [██████░░

In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  CELL 9 — Final Test Evaluation                             │
# └─────────────────────────────────────────────────────────────┘

print(f"\nLoading best checkpoint: {BEST_MODEL_PATH}")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.unfreeze_all()

e_test = evaluate(loaders["evasion_test"], EVASION_LABELS)
s_test = evaluate(loaders["stance_test"],  STANCE_LABELS)

print_eval(e_test, "QA EVASION DETECTION — FINAL TEST RESULTS")
print_eval(s_test, "STANCE DETECTION — FINAL TEST RESULTS")

pd.DataFrame({
    "text":       Xete,
    "true_label": [EVASION_LABELS[l] for l in e_test["labels"]],
    "pred_label": [EVASION_LABELS[p] for p in e_test["preds"]],
}).to_csv("results_evasion.csv", index=False)

pd.DataFrame({
    "text":       Xste,
    "true_label": [STANCE_LABELS[l] for l in s_test["labels"]],
    "pred_label": [STANCE_LABELS[p] for p in s_test["preds"]],
}).to_csv("results_stance.csv", index=False)

print("✓ results_evasion.csv saved")
print("✓ results_stance.csv saved")
print(f"✓ {BEST_MODEL_PATH} saved")



In [ ]:
from google.colab import files
files.download("best_model.pt")
files.download("results_evasion.csv")
files.download("results_stance.csv")
